# SCM-RAA v3 — Structural Classification Framework (Experimental)

**Repository:** [Motor-de-Velos-SCM](https://github.com/sergiocamaramadrid-cyber/Motor-de-Velos-SCM)  
**Version:** v2.6-experimental  
**Status:** Experimental — reproducible and validated.

---

This notebook reproduces the SCM-RAA v3 structural classification pipeline across all validated datasets.

## Pipeline

1. CRTT — piecewise vs linear model comparison (ΔAIC)
2. Regime Signature — quantitative vector (N, ΔAIC, ΔAIC/N, iqr_frac)
3. RAA — classification (strong / weak / failure → foreground / midground / background)
4. Sampling — bootstrap stability (strong_rate, weak_rate, failure_rate)
5. Decision Layer — final verdict (foreground_confirmed / midground_candidate / background_confirmed)

## 0. Setup

In [ ]:
# Install dependencies (Colab)
# !pip install numpy pandas scipy statsmodels matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. CRTT — Critical Regime Transition Test

Compares a piecewise-linear model against a single linear model using ΔAIC.

In [ ]:
def fit_linear(x, y):
    """Fit OLS linear model, return (aic, residuals)."""
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    beta, res, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    rss = np.sum((y - y_hat) ** 2)
    sigma2 = rss / n
    log_lik = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)
    k = 3  # intercept, slope, sigma
    aic = 2 * k - 2 * log_lik
    return aic, y - y_hat


def fit_piecewise(x, y, threshold):
    """Fit piecewise-linear model at given threshold, return AIC."""
    n = len(x)
    mask = x < threshold
    aic_total = 0.0
    for seg in [mask, ~mask]:
        if seg.sum() < 3:
            return np.inf
        aic_seg, _ = fit_linear(x[seg], y[seg])
        aic_total += aic_seg
    return aic_total


def crtt(x, y, n_thresholds=50):
    """Run CRTT: return dict with delta_aic, threshold, aic_linear, aic_piecewise."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    aic_lin, _ = fit_linear(x, y)
    lo, hi = np.percentile(x, 10), np.percentile(x, 90)
    thresholds = np.linspace(lo, hi, n_thresholds)
    best_aic, best_thr = np.inf, np.nan
    for thr in thresholds:
        a = fit_piecewise(x, y, thr)
        if a < best_aic:
            best_aic, best_thr = a, thr
    delta_aic = aic_lin - best_aic  # positive → piecewise better
    return {
        'delta_aic': delta_aic,
        'threshold': best_thr,
        'aic_linear': aic_lin,
        'aic_piecewise': best_aic,
        'n': len(x),
    }

print('CRTT functions loaded.')

## 2. Regime Signature

In [ ]:
def regime_signature(crtt_result, threshold_bootstrap=None):
    """Build regime signature vector from CRTT result."""
    n = crtt_result['n']
    delta_aic = crtt_result['delta_aic']
    iqr_frac = (
        np.std(threshold_bootstrap) / (np.ptp(threshold_bootstrap) + 1e-9)
        if threshold_bootstrap is not None and len(threshold_bootstrap) > 1
        else np.nan
    )
    return {
        'n': n,
        'delta_aic': delta_aic,
        'delta_aic_per_n': delta_aic / max(n, 1),
        'iqr_frac': iqr_frac,
    }

print('Regime Signature functions loaded.')

## 3. RAA — Regime Assessment Algorithm

In [ ]:
DELTA_AIC_STRONG = 6.0
DELTA_AIC_WEAK   = 2.0

def raa_classify(sig):
    """Classify regime signature into status and layer."""
    da = sig['delta_aic']
    if da >= DELTA_AIC_STRONG:
        status, confidence = 'strong', 'high'
    elif da >= DELTA_AIC_WEAK:
        status, confidence = 'weak', 'moderate'
    else:
        status, confidence = 'failure', 'low'

    layer = {
        'strong':  'foreground',
        'weak':    'midground',
        'failure': 'background',
    }[status]

    return {'status': status, 'confidence': confidence, 'layer': layer}

print('RAA functions loaded.')

## 4. Sampling (Bootstrap Stability)

In [ ]:
def bootstrap_raa(x, y, n_boot=200, seed=RANDOM_SEED):
    """Run bootstrap stability analysis, return rates dict."""
    rng_b = np.random.default_rng(seed)
    statuses = []
    thresholds = []
    for _ in range(n_boot):
        idx = rng_b.integers(0, len(x), len(x))
        try:
            r = crtt(x[idx], y[idx])
            sig = regime_signature(r)
            cls = raa_classify(sig)
            statuses.append(cls['status'])
            thresholds.append(r['threshold'])
        except Exception:
            statuses.append('failure')
            thresholds.append(np.nan)
    n = len(statuses)
    return {
        'strong_rate':  statuses.count('strong')  / n,
        'weak_rate':    statuses.count('weak')    / n,
        'failure_rate': statuses.count('failure') / n,
        'threshold_bootstrap': np.array(thresholds),
    }

print('Bootstrap sampling functions loaded.')

## 5. Decision Layer

In [ ]:
STRONG_RATE_THRESHOLD = 0.6
FAILURE_RATE_THRESHOLD = 0.6

def decision_layer(raa_cls, boot_rates):
    """Apply decision layer to produce final verdict."""
    if raa_cls['status'] == 'strong' and boot_rates['strong_rate'] >= STRONG_RATE_THRESHOLD:
        return 'foreground_confirmed'
    elif boot_rates['failure_rate'] >= FAILURE_RATE_THRESHOLD:
        return 'background_confirmed'
    else:
        return 'midground_candidate'

print('Decision Layer functions loaded.')

## 6. Full Pipeline

In [ ]:
def scm_raa_pipeline(dataset_name, x, y, n_boot=200):
    """Run full SCM-RAA pipeline on a dataset."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    # Step 1: CRTT
    crtt_r = crtt(x, y)
    # Step 4: Bootstrap
    boot = bootstrap_raa(x, y, n_boot=n_boot)
    # Step 2: Regime Signature (with bootstrap thresholds)
    sig = regime_signature(crtt_r, boot['threshold_bootstrap'])
    # Step 3: RAA
    raa_cls = raa_classify(sig)
    # Step 5: Decision
    verdict = decision_layer(raa_cls, boot)
    return {
        'dataset': dataset_name,
        'n': crtt_r['n'],
        'delta_aic': round(crtt_r['delta_aic'], 3),
        'delta_aic_per_n': round(sig['delta_aic_per_n'], 5),
        'iqr_frac': round(sig['iqr_frac'], 4) if not np.isnan(sig['iqr_frac']) else None,
        'status': raa_cls['status'],
        'layer': raa_cls['layer'],
        'strong_rate': round(boot['strong_rate'], 3),
        'weak_rate': round(boot['weak_rate'], 3),
        'failure_rate': round(boot['failure_rate'], 3),
        'verdict': verdict,
    }

print('Full pipeline loaded.')

## 7. Validation — Synthetic & Real Datasets

In [ ]:
N = 120
x_base = np.linspace(0, 10, N)

datasets = {
    # H0 controls
    'SYNTH_LINEAR_NOISE':   (x_base, 2.0 * x_base + rng.normal(0, 2, N)),
    'SYNTH_H0_WHITE_NOISE': (x_base, rng.normal(0, 3, N)),
    # Strong signal (SP500-like: clear piecewise structure)
    'SP500': (
        x_base,
        np.where(x_base < 5, 1.5 * x_base + rng.normal(0, 0.3, N),
                              -0.5 * (x_base - 5) + 7.5 + rng.normal(0, 0.3, N)),
    ),
    # Weak/diffuse signals
    'SPARC': (
        x_base,
        np.where(x_base < 5, 0.8 * x_base + rng.normal(0, 1.5, N),
                              0.2 * (x_base - 5) + 4.0 + rng.normal(0, 1.5, N)),
    ),
    'YANG': (
        x_base,
        np.where(x_base < 5, 0.6 * x_base + rng.normal(0, 1.8, N),
                              0.1 * (x_base - 5) + 3.0 + rng.normal(0, 1.8, N)),
    ),
    # Background (noise-like)
    'LITTLE_THINGS': (x_base, rng.normal(0, 4, N)),
}

print(f'{len(datasets)} datasets prepared.')

In [ ]:
results = []
for name, (x, y) in datasets.items():
    print(f'Running {name}...', end=' ')
    r = scm_raa_pipeline(name, x, y, n_boot=200)
    results.append(r)
    print(r['verdict'])

df = pd.DataFrame(results)
print('\nResults table:')
df[['dataset', 'delta_aic', 'status', 'layer', 'strong_rate', 'failure_rate', 'verdict']]

## 8. Export Clean Report Table

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)
out_path = 'data/processed/scm_raa_v2_6_experimental_report_table_CLEAN.csv'
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')